In [ ]:
from pathlib import Path

import os
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import pandas as pd

import time
from pathlib import Path

# Project root: Saltare/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = PROJECT_ROOT / "models" / "pose_landmarker_full.task"

VIDEO_NAMES = os.listdir(PROJECT_ROOT / "data")

# take random video from data folder for input path

INPUT_PATH = PROJECT_ROOT / "data" / np.random.choice(VIDEO_NAMES)
OUTPUT_DIR = PROJECT_ROOT / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Input:", INPUT_PATH)
print("Model:", MODEL_PATH)
print("Output:", OUTPUT_DIR)

Project: c:\Users\louiz\Documents\Projects\Saltare
Input: c:\Users\louiz\Documents\Projects\Saltare\data\dance_video_1.mp4
Model: c:\Users\louiz\Documents\Projects\Saltare\models\pose_landmarker_full.task
Output: c:\Users\louiz\Documents\Projects\Saltare\output


In [2]:
JOINTS = {
    "nose": 0,
    "left_shoulder": 11,
    "right_shoulder": 12,
    "left_elbow": 13,
    "right_elbow": 14,
    "left_wrist": 15,
    "right_wrist": 16,
    "left_hip": 23,
    "right_hip": 24,
    "left_knee": 25,
    "right_knee": 26,
    "left_ankle": 27,
    "right_ankle": 28,
    "left_foot": 31,
    "right_foot": 32,
}

CONNECTIONS = [
    ("nose", "left_shoulder"),
    ("nose", "right_shoulder"),
    ("left_shoulder", "right_shoulder"),
    ("left_shoulder", "left_elbow"),
    ("left_elbow", "left_wrist"),
    ("right_shoulder", "right_elbow"),
    ("right_elbow", "right_wrist"),
    ("left_shoulder", "left_hip"),
    ("right_shoulder", "right_hip"),
    ("left_hip", "right_hip"),
    ("left_hip", "left_knee"),
    ("left_knee", "left_ankle"),
    ("left_ankle", "left_foot"),
    ("right_hip", "right_knee"),
    ("right_knee", "right_ankle"),
    ("right_ankle", "right_foot"),
]

# OpenCV uses BGR colors.
JOINT_COLOR = (0, 255, 0)
BONE_COLOR = (0, 200, 255)

JOINT_RADIUS = 4
BONE_THICKNESS = 2
MIN_VISIBILITY = 0.5

print(f"Configured {len(JOINTS)} joints.")

Configured 15 joints.


In [3]:
options = vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(
        model_asset_path=str(MODEL_PATH)
    ),
    running_mode=vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

print("Pose Landmarker configured.")

Pose Landmarker configured.


In [4]:
frames_rgb = []
skeletons = []
timestamps_ms = []

cap = cv2.VideoCapture(str(INPUT_PATH))

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if fps <= 0:
    cap.release()
    raise RuntimeError("Invalid video frame rate.")

joint_indices = list(JOINTS.values())

try:
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        frame_index = 0

        while True:
            success, frame_bgr = cap.read()

            if not success:
                break

            frame_rgb = cv2.cvtColor(
                frame_bgr, cv2.COLOR_BGR2RGB
            )

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=np.ascontiguousarray(frame_rgb),
            )

            timestamp_ms = round(frame_index * 1000 / fps)

            result = landmarker.detect_for_video(
                mp_image,
                timestamp_ms,
            )

            # Shape: (15 joints, 3 values)
            pose = np.full(
                (len(JOINTS), 3),
                np.nan,
                dtype=np.float32,
            )

            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]

                for j, landmark_idx in enumerate(joint_indices):
                    landmark = landmarks[landmark_idx]
                    visibility = landmark.visibility

                    if (
                        visibility is not None
                        and visibility >= MIN_VISIBILITY
                        and 0 <= landmark.x <= 1
                        and 0 <= landmark.y <= 1
                    ):
                        pose[j] = [
                            landmark.x,
                            landmark.y,
                            visibility,
                        ]

            skeletons.append(pose)
            timestamps_ms.append(timestamp_ms)

            frame_index += 1

            if frame_index % 30 == 0:
                print(f"Processed {frame_index}/{total_frames} frames")

finally:
    cap.release()

if len(skeletons) == 0:
    raise RuntimeError("No frames were processed from the input video.")

skeletons = np.asarray(skeletons, dtype=np.float32)
timestamps_ms = np.asarray(timestamps_ms, dtype=np.int64)

frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

print("\nPose extraction complete.")
print(f"Frames processed: {len(skeletons)}")
print(f"Pose array shape: {skeletons.shape}")
print(f"Frame size: {frame_width} x {frame_height}")

Processed 30/440 frames
Processed 60/440 frames
Processed 90/440 frames
Processed 120/440 frames
Processed 150/440 frames
Processed 180/440 frames
Processed 210/440 frames
Processed 240/440 frames
Processed 270/440 frames
Processed 300/440 frames
Processed 330/440 frames
Processed 360/440 frames
Processed 390/440 frames
Processed 420/440 frames

Pose extraction complete.
Frames processed: 437
Pose array shape: (437, 15, 3)
Frame size: -1 x -1


In [5]:
import json

POSE_PATH = OUTPUT_DIR / "test_video_pose.npz"
METADATA_PATH = OUTPUT_DIR / "test_video_pose_metadata.json"

# Copy normalized coordinates and convert x/y to pixel coordinates.
keypoints_px = skeletons.copy()
keypoints_px[:, :, 0] *= frame_width
keypoints_px[:, :, 1] *= frame_height

joint_names = list(JOINTS.keys())

np.savez_compressed(
    POSE_PATH,
    keypoints_normalized=skeletons,
    keypoints_px=keypoints_px,
    timestamps_ms=timestamps_ms,
    fps=np.float32(fps),
    joint_names=np.asarray(joint_names),
)

metadata = {
    "source_video": INPUT_PATH.name,
    "num_frames": len(skeletons),
    "fps": float(fps),
    "frame_width": frame_width,
    "frame_height": frame_height,
    "num_joints": len(joint_names),
    "joint_names": joint_names,
    "connections": [list(edge) for edge in CONNECTIONS],
    "coordinate_formats": {
        "keypoints_normalized": "[x, y, visibility]; x/y normalized to 0-1",
        "keypoints_px": "[x, y, visibility]; x/y measured in pixels",
    },
    "missing_joint_convention": "NaN values",
}

METADATA_PATH.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print(f"Saved pose data: {POSE_PATH}")
print(f"Saved metadata:  {METADATA_PATH}")
print(f"Pose array shape: {skeletons.shape}")

Saved pose data: c:\Users\louiz\Documents\Projects\Saltare\output\test_video_pose.npz
Saved metadata:  c:\Users\louiz\Documents\Projects\Saltare\output\test_video_pose_metadata.json
Pose array shape: (437, 15, 3)


In [6]:
data = np.load(POSE_PATH)

keypoints = data["keypoints_normalized"]
keypoints_px = data["keypoints_px"]
timestamps = data["timestamps_ms"]
joint_names = data["joint_names"].tolist()

print("Number of frames:", keypoints.shape[0])
print("Number of joints:", keypoints.shape[1])
print("Values per joint: x, y, visibility")
print("Joint names:", joint_names)

# Show the first frame's joint coordinates.

first_frame_df = pd.DataFrame(
    keypoints[0],
    columns=["x", "y", "visibility"],
    index=joint_names,
)

display(first_frame_df)

Number of frames: 437
Number of joints: 15
Values per joint: x, y, visibility
Joint names: ['nose', 'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle', 'left_foot', 'right_foot']


,x,y,visibility
nose,0.498486,0.136549,0.999979
left_shoulder,0.600072,0.228722,0.999973
right_shoulder,0.380899,0.220937,0.999816
left_elbow,0.610014,0.329390,0.918798
right_elbow,0.366307,0.317451,0.893991
left_wrist,0.655955,0.413256,0.916757
right_wrist,0.308933,0.382850,0.863033
left_hip,0.540783,0.394381,0.999614
right_hip,0.411913,0.401730,0.999406
left_knee,0.523972,0.532472,0.975608


In [ ]:
def draw_skeleton(frame_rgb, pose):
    frame = cv2.cvtColor(
        frame_rgb.copy(),
        cv2.COLOR_RGB2BGR,
    )

    height, width = frame.shape[:2]
    points = {}

    for i, name in enumerate(JOINTS.keys()):
        x, y, visibility = pose[i]

        if not np.isfinite(x) or not np.isfinite(y):
            continue

        px = int(round(x * width))
        py = int(round(y * height))

        if 0 <= px < width and 0 <= py < height:
            points[name] = (px, py)

    # Draw bones first.
    for start, end in CONNECTIONS:
        if start in points and end in points:
            cv2.line(
                frame,
                points[start],
                points[end],
                BONE_COLOR,
                BONE_THICKNESS,
                cv2.LINE_AA,
            )

    # Draw joints over the bones.
    for px, py in points.values():
        cv2.circle(
            frame,
            (px, py),
            JOINT_RADIUS,
            JOINT_COLOR,
            -1,
            cv2.LINE_AA,
        )

    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

In [ ]:

from turtle import width


def draw_skeleton_bgr(frame, pose):
    """Draw a cute stick figure with thick limbs and a blank round head."""
    height, width = frame.shape[:2]
    points = {}

    # Colors are in BGR format.
    LIMB_COLOR =  (246, 242, 216)
    JOINT_COLOR = (255, 255, 255)
    HEAD_COLOR = (255, 255, 255)

    LIMB_THICKNESS = max(20, width // 60)
    JOINT_RADIUS = max(10, width // 60)
    HEAD_RADIUS = max(75, width // 14)

    # Convert normalized coordinates to pixel coordinates.
    for i, name in enumerate(JOINTS):
        x, y, visibility = pose[i]

        if not np.isfinite(x) or not np.isfinite(y):
            continue

        px = int(round(x * width))
        py = int(round(y * height))

        if 0 <= px < width and 0 <= py < height:
            points[name] = (px, py)

    # Skip the nose-to-shoulder connections.
    hidden_connections = {
        frozenset(("nose", "left_shoulder")),
        frozenset(("nose", "right_shoulder")),
    }

    # Draw thick limbs with rounded ends.
    for start, end in CONNECTIONS:
        if frozenset((start, end)) in hidden_connections:
            continue

        if start in points and end in points:
            p1 = points[start]
            p2 = points[end]

            cv2.line(
                frame, p1, p2,
                LIMB_COLOR, LIMB_THICKNESS, cv2.LINE_AA
            )

            # Round off both ends of each limb.
            cv2.circle(
                frame, p1, LIMB_THICKNESS // 2,
                LIMB_COLOR, -1, cv2.LINE_AA
            )
            cv2.circle(
                frame, p2, LIMB_THICKNESS // 2,
                LIMB_COLOR, -1, cv2.LINE_AA
            )

    # Draw soft, round joints.
    for px, py in points.values():
        cv2.circle(
            frame, (px, py), JOINT_RADIUS,
            JOINT_COLOR, -1, cv2.LINE_AA
        )

    # Position the large blank head using the ears or nose.
    if "left_ear" in points and "right_ear" in points:
        head_x = (points["left_ear"][0] + points["right_ear"][0]) // 2
        head_y = (points["left_ear"][1] + points["right_ear"][1]) // 2
    elif "nose" in points:
        head_x, head_y = points["nose"]
    elif "left_shoulder" in points and "right_shoulder" in points:
        head_x = (points["left_shoulder"][0] + points["right_shoulder"][0]) // 2
        head_y = (points["left_shoulder"][1] + points["right_shoulder"][1]) // 2 - HEAD_RADIUS * 2
    else:
        head_x = head_y = None

    if head_x is not None:
        cv2.circle(
            frame, (head_x, head_y), HEAD_RADIUS,
            HEAD_COLOR, -1, cv2.LINE_AA
        )

    return frame


In [28]:

def play_pose_stick_figure(
    video_path,
    pose_path,
    window_name="Saltare - Stick Figure",
):
    """
    Display only the animated stick figure on a black background.
    Uses saved pose data; does not rerun pose estimation
    or save a rendered video.
    Press Q or Esc to exit.
    """
    video_path = Path(video_path)
    pose_path = Path(pose_path)

    # Load saved pose data.
    with np.load(pose_path) as data:
        poses = data["keypoints_normalized"].copy()
        saved_fps = float(data["fps"])

    # Read video metadata only; don't display its frames.
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    fps = video_fps if video_fps > 0 else saved_fps

    if fps <= 0:
        fps = 30.0

    frame_interval = 1.0 / fps
    frame_index = 0
    start_time = time.perf_counter()

    try:
        while frame_index < len(poses):
            # Create a blank black canvas for each frame.
            canvas = np.zeros(
                (frame_height, frame_width, 3),
                dtype=np.uint8,
            )

            # Draw only the stick figure.
            draw_skeleton_bgr(
                canvas,
                poses[frame_index],
            )

            cv2.imshow(window_name, canvas)

            # Maintain the target playback speed.
            target_time = (
                start_time + (frame_index + 1) * frame_interval
            )
            remaining = target_time - time.perf_counter()
            delay_ms = max(1, round(remaining * 1000))

            key = cv2.waitKey(delay_ms) & 0xFF

            if key in (ord("q"), 27):
                break

            frame_index += 1

    finally:
        cv2.destroyAllWindows()

    print(
        f"Displayed {frame_index} stick-figure frames "
        f"at a target of {fps:.2f} FPS."
    )


# Display the stick figure using your existing saved pose data.
play_pose_stick_figure(
    INPUT_PATH,
    POSE_PATH,
)


KeyboardInterrupt: 

In [ ]:
play_pose_stick_figure(INPUT_PATH, POSE_PATH)

KeyboardInterrupt: 